# AgriNav — RiceSEG Backbone Pretraining (Colab / GPU)

Runs the project's **first training batch**: pretrain the WeedDet Det-ResNet-50 backbone on RiceSEG segmentation, then export a backbone the detector reuses.

This notebook only *drives* `training/riceseg_pretrain.py` from the repo — it defines no model, loss, or training logic of its own (per the deployment roadmap: notebooks are disposable views, code is the source of truth).

**Before you run:** `Runtime → Change runtime type → GPU`, and put `RiceSEG.zip` somewhere in your Google Drive (default expected: `MyDrive/agrinav_data/RiceSEG.zip`).

Outputs (backbone `.pth`, full checkpoint, and `manifest.json`) are written back to `MyDrive/agrinav_data/out/`.

## 1. Confirm GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code (clone the repo)

**This repository is PRIVATE**, so an anonymous clone fails with
`could not read Username for 'https://github.com'` — Colab has no interactive prompt.

Pick ONE of:

1. **Colab secret (recommended).** Left sidebar → 🔑 Secrets → add `GITHUB_TOKEN` holding a
   fine-grained GitHub PAT with **Contents: Read** on this repo, and toggle notebook access on.
   The cell below reads it from Colab's secret store: the token is never printed, never saved
   into the notebook, and is stripped from the git remote after cloning.
2. **Make the repo public** — then this cell works with no token at all.
3. **Skip GitHub:** upload the repo folder to Drive and set
   `REPO_DIR = '/content/drive/MyDrive/agrinav_repo'`.

The cell **stops immediately** if the code isn't available, so later cells can't produce
confusing follow-on errors.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/Bmerrysmith/Autonomous-tractor-system.git'
BRANCH   = 'codex/repository-recovery'
REPO_DIR = '/content/agrinav'

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
print('GitHub token found in Colab secrets:', bool(token))

def _redact(s):
    return s.replace(token, '***') if token else s

# GIT_TERMINAL_PROMPT=0 turns an auth failure into an immediate error, not a hang.
env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '--branch', BRANCH, auth_url, REPO_DIR],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            'CLONE FAILED — stopping so later cells do not cascade.\n\n'
            + _redact(r.stderr) +
            '\nFix: add a GITHUB_TOKEN Colab secret (fine-grained PAT, Contents:Read), '
            'or make the repo public, or upload the repo to Drive and set REPO_DIR.')
    # Never persist the token in .git/config
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL], env=env)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], env=env)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], env=env)

os.chdir(REPO_DIR)
assert os.path.exists('training/riceseg_pretrain.py'), (
    f'Repo at {REPO_DIR} has no training/riceseg_pretrain.py — wrong branch or upload?')
print('repo ready at', REPO_DIR)
!git log --oneline -1

## 4. Install dependencies

In [ ]:
# Colab already ships torch+torchvision with CUDA; install the rest.
!pip install -q numpy Pillow tqdm

## 5. Locate and extract RiceSEG from Drive

Verifies the archive SHA-256 against the project registry before extracting, and extracts to fast local Colab disk (`/content`), **not** Drive.

In [ ]:
import hashlib, zipfile, glob, os

# Where you put RiceSEG.zip in Drive (edit if different):
RICESEG_ZIP = '/content/drive/MyDrive/agrinav_data/RiceSEG.zip'
EXPECTED_SHA = '0071d9f941508afd9a86aa5ef740433dee938e1c7a9508a191d3e46a2909be96'
DATA_ROOT = '/content/RiceSEG'

if not os.path.exists(RICESEG_ZIP):
    hits = glob.glob('/content/drive/MyDrive/**/RiceSEG.zip', recursive=True)
    assert hits, 'RiceSEG.zip not found in Drive. Upload it and set RICESEG_ZIP.'
    RICESEG_ZIP = hits[0]
print('using', RICESEG_ZIP)

h = hashlib.sha256()
with open(RICESEG_ZIP, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
print('sha256:', h.hexdigest())
assert h.hexdigest() == EXPECTED_SHA, 'RiceSEG.zip hash mismatch — do not train on an unknown archive.'

os.makedirs(DATA_ROOT, exist_ok=True)
with zipfile.ZipFile(RICESEG_ZIP) as z:
    z.extractall(DATA_ROOT)
print('extracted to', DATA_ROOT)
print(os.listdir(os.path.join(DATA_ROOT, 'global rice segmentation')))

## 6. Sanity gates: self-test + tiny overfit

`--self-test` checks the backbone key contract. The overfit gate must reach mIoU ≥ 0.80 on 8 all-class tiles or it **exits non-zero** — that means the pipeline is broken and you must stop before the full run. It writes only to a temp path, never the production backbone.

In [ ]:
!python -B training/riceseg_pretrain.py --self-test
!python -B training/riceseg_pretrain.py --data-root "{DATA_ROOT}" --overfit 8 --batch-size 4 --img-size 512

## 7. Full pretraining (ImageNet → RiceSEG)

Production config: 512px, 30 epochs, ImageNet warm-start (the real `ImageNet→RiceSEG` condition). Writes the backbone, a full resumable checkpoint, and an immutable `manifest.json` to Drive. Expect roughly 30–60 min on a T4/A100.

For the `random→RiceSEG` control condition, add `--no-imagenet`. For a country holdout, add e.g. `--holdout-country Tanzania`.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/agrinav_data/out'
os.makedirs(OUT_DIR, exist_ok=True)
OUT = OUT_DIR + '/riceseg_backbone.pth'

!python -B training/riceseg_pretrain.py \
  --data-root "{DATA_ROOT}" \
  --epochs 30 --batch-size 12 --img-size 512 --lr 3e-4 --val-ratio 0.1 --seed 42 \
  --out "{OUT}"

## 8. Inspect results

The manifest records git commit, environment, ImageNet coverage, per-epoch history, best epoch/mIoU, per-class IoU, and the backbone sha256. Paste this into `docs/research/RICESEG_PRETRAIN_RESULTS.md`.

In [ ]:
import json
m = json.load(open(OUT + '.manifest.json'))
print('git_commit      :', m['git_commit'])
print('environment     :', m['environment'])
print('imagenet_coverage:', m['imagenet_coverage'])
print('best epoch/mIoU :', m['best']['epoch'], round(m['best']['miou_present_classes'], 4))
print('per-class IoU   :', m['best']['per_class_iou'])
print('absent classes  :', m['best']['absent_classes'])
print('backbone sha256 :', m['backbone_export']['sha256'])
print()
print('Artifacts in', OUT_DIR, ':')
print(' ', OUT.split('/')[-1], '(backbone for WeedDet)')
print(' ', OUT.split('/')[-1] + '.fullckpt.pth (resumable)')
print(' ', OUT.split('/')[-1] + '.manifest.json (run record)')

## Next steps

1. Copy the three printed metrics blocks into `docs/research/RICESEG_PRETRAIN_RESULTS.md` (or send them back).
2. The backbone at `MyDrive/agrinav_data/out/riceseg_backbone.pth` is what the detector loads via `load_riceseg_backbone()`.
3. Detector training comes next — but its code fixes (`models/weeddet_v6b.py`) and the reviewed weed masks are still open; see `docs/detector_dataset_card.md`.